In [83]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()
print(os.getenv("OPENAI_API_KEY") is not None)

In [ ]:
llm = ChatOpenAI(model_name="gpt-5-nano",  
 temperature=0.7)

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    #take user query from state
    messages = state["messages"]
    #send to llm
    response = llm.invoke(messages)
    # response store state
    return {'messages':[response]}

In [87]:
graph = StateGraph(ChatState)

#add nodes
graph.add_node("chat_node",chat_node)

#add edges
graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

chatbot = graph.compile()


In [ ]:
chatbot

In [ ]:
initialstate = {'messages':[HumanMessage(content="What is quantum computing?")]}

In [67]:
response = chatbot.invoke(initialstate)

In [80]:
response['messages'][-1].content


'Quantum computing is a way of processing information that uses the rules of quantum mechanics instead of classical physics. It relies on special units called qubits.\n\nKey ideas:\n- Qubits: A classical bit is 0 or 1. A qubit can be in a mixture of both (a superposition): written roughly as α|0> + β|1>, with probabilities |α|^2 and |β|^2.\n- Superposition: Allows a quantum computer to explore many possible states at once.\n- Entanglement: Qubits can become linked so the state of one instantly affects the state of another, no matter how far apart they are. This creates strong correlations that classical systems can’t reproduce.\n- Interference: Quantum states interfere with each other so that correct answers are amplified and wrong ones are diminished.\n\nHow it works at a high level:\n- Quantum gates manipulate qubits with operations similar to logic gates in classical computers, but they are reversible and operate on superpositions.\n- A quantum circuit is a sequence of these gates.\

In [82]:
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    initialstate = {'messages':[HumanMessage(content=user_input)]}
    response = chatbot.invoke(initialstate)
    print("AI:", response['messages'][-1].content)
    

AI: Probability is a way to measure how likely it is that something will happen. It assigns a number between 0 and 1 to an event, with 0 meaning “impossible” and 1 meaning “certain.”

Key ideas:
- Interpretations
  - Classical: probability = number of favorable outcomes / total number of equally likely outcomes.
  - Frequentist: probability is the long-run average frequency of the event if you repeat the experiment many times.
  - Subjective: probability is a degree of belief about how likely the event is.
- Basic rules
  - 0 ≤ P(A) ≤ 1 for any event A.
  - If S is the sample space, P(S) = 1.
  - Complement: P(A) = 1 − P(A’s complement).
  - Addition (union): If A and B are disjoint, P(A ∪ B) = P(A) + P(B). In general, P(A ∪ B) = P(A) + P(B) − P(A ∩ B).
  - Multiplication (independence): If A and B are independent, P(A ∩ B) = P(A)P(B). More generally, P(A ∩ B) = P(A)P(B|A).
- Simple examples
  - Coin flip: P(heads) = 1/2 = 0.5.
  - Deck of cards: P(drawing an ace) = 4/52 = 1/13.
  - Di

### With Check point,

In [88]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)

#add nodes
graph.add_node("chat_node",chat_node)

#add edges
graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

chatbot = graph.compile(checkpointer=checkpoint)


In [90]:
thread_id = "1"

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
     
    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_input)]}, config=config)
    print("AI:", response['messages'][-1].content)
    

AI: Your name is Udayakumar.

Would you like me to call you Uday or keep using Udayakumar? Also, should I remember your name for this session or for future interactions (memory availability depends on your platform)? How can I help you next?


In [93]:
database = chatbot.get_state(config=config)

In [95]:
print(database)

StateSnapshot(values={'messages': [HumanMessage(content='My name is udayakumar', additional_kwargs={}, response_metadata={}, id='8c5566fb-e8d1-4700-947d-d147fce81f19'), AIMessage(content='Nice to meet you, Udayakumar! How would you like me to address you? What would you like help with today? I can assist with things like writing and editing, learning topics, coding, planning, translations, or just answering questions. Tell me a goal or task, and we can get started.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 649, 'prompt_tokens': 13, 'total_tokens': 662, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxXHxrLYk2xxbeB98P0ND8lmOfJ5n', 'service_tier': 'default', 'finish_re